In [1]:
import logging
import benchmarklib
from benchmarklib import BenchmarkDatabase, CompileType
from benchmarklib.algorithms import GroverRunner, GroverConfig, calculate_grover_iterations
from benchmarklib.compilers import TruthTableCompiler, XAGCompiler, QCFCompiler
from benchmarklib.pipeline import PipelineCompiler
from benchmarklib.pipeline.synthesis import QuantumMPC

from sqlalchemy import select, func
from sqlalchemy.orm import joinedload

from sat import ThreeSat, ThreeSatTrial, ThreeSatOracleTrial, populate_problems, run_three_sat_benchmark, run_three_sat_benchmark_sample, logger as three_sat_logger

benchmarklib.setup_logging(logging.INFO)

three_sat_logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
handler.setLevel(logging.INFO)
three_sat_logger.addHandler(handler)
three_sat_logger.propagate = False

from compiler_configs import compiler_qmpc_ai_test, compiler_qmpc_standard, compiler_qcf, compiler_xag

db = BenchmarkDatabase("three_sat.db", ThreeSat, ThreeSatTrial)
oracle_db = BenchmarkDatabase("three_sat.db", ThreeSat, ThreeSatOracleTrial)

benchmarklib.core.database - INFO - Database initialized: three_sat.db (<property object at 0x78183a515c10>)
benchmarklib.core.database - INFO - Database initialized: three_sat.db (<property object at 0x78183a515c10>)


In [ ]:
populate_problems(db, var_range=range(3, 5), num_problems=300)

In [ ]:
for compiler in [compiler_qcf,compiler_xag]:
    backend = compiler.config.backend
    run_three_sat_benchmark(db, compiler, backend, num_vars_iter=range(3, 11), num_problems=20)

In [ ]:
for compiler in [compiler_xag, compiler_qcf,]: #
    backend = compiler.config.backend
    run_three_sat_benchmark(oracle_db, compiler, backend, num_vars_iter=range(3, 5), num_problems=300, save_circuits=True)

### Cherrypick problems

In [ ]:
problems = []
for n in range(3, 13):
    for p in db.query(select(ThreeSat).where(ThreeSat.num_vars == n)):
        n_problems = []
        if len(n_problems) < 10 and len(p.solutions) == 0:
            n_problems.append(p)
        for problem in n_problems:
            if len(db.query(
                select(ThreeSatOracleTrial.id)
                .where(ThreeSatOracleTrial.problem_id == problem.id)
                .where(ThreeSatOracleTrial.compiler_name == "CLASSICAL_FUNCTION")
                )) == 0:
                problems.append(problem)

In [ ]:
len(problems)

In [ ]:
for compiler in [compiler_qcf,  ]: #compiler_xag,
    backend = compiler.config.backend
    print(f"Running sample benchmark for compiler {compiler.name} on {len(problems)} problems.")
    run_three_sat_benchmark_sample(oracle_db, compiler, backend, problems, save_circuits=True)

In [ ]:
p = problems[0]
print(p.num_vars, len(p.solutions))

In [ ]:
result = compiler_qcf.compile(p)
result.synthesis_circuit.draw()

### Collect Compile Times

In [2]:
from sqlalchemy import intersect, union
from sqlalchemy.orm import joinedload
import datetime

def collect_compile_times(db, n_iter=range(3, 9), samples_per_n=100):
    for n in n_iter:
        available_problem_ids = db.query(
        intersect(
                select(db.trial_class.problem_id).join(db.trial_class.problem).where(db.problem_class.num_vars == n).where(db.trial_class.compiler_name == "CLASSICAL_FUNCTION").where(db.trial_class.is_failed==False),
                select(db.trial_class.problem_id).join(db.trial_class.problem).where(db.problem_class.num_vars == n).where(db.trial_class.compiler_name == "XAG").where(db.trial_class.is_failed==False)
            )
        )
        if len(available_problem_ids) == 0:
            available_problem_ids = db.query(
                union(
                    select(db.trial_class.problem_id).join(db.trial_class.problem).where(db.problem_class.num_vars == n).where(db.trial_class.compiler_name == "CLASSICAL_FUNCTION").where(db.trial_class.is_failed==False).limit(samples_per_n).subquery().select(),
                    select(db.trial_class.problem_id).join(db.trial_class.problem).where(db.problem_class.num_vars == n).where(db.trial_class.compiler_name == "XAG").where(db.trial_class.is_failed==False).limit(samples_per_n).subquery().select()
                )
            )
        else:
            available_problem_ids = available_problem_ids[:samples_per_n]

        print(f"Num problems: {len(available_problem_ids)}")

        for problem_id in available_problem_ids:
            trial_xag = db.query(select(ThreeSatOracleTrial).where(ThreeSatOracleTrial.problem_id == problem_id, ThreeSatOracleTrial.compiler_name == "XAG").limit(1).options(joinedload(ThreeSatOracleTrial.problem)))
            trial_esop = db.query(select(ThreeSatOracleTrial).where(ThreeSatOracleTrial.problem_id == problem_id, ThreeSatOracleTrial.compiler_name == "CLASSICAL_FUNCTION").limit(1).options(joinedload(ThreeSatOracleTrial.problem)))

            for _trial, compiler in [(trial_xag, compiler_xag), (trial_esop, compiler_qcf)]:
                if len(_trial) == 0:
                    continue
                trial = _trial[0]

                if trial.extra_data is not None and trial.extra_data.get("compile_time") is not None:
                    continue

                start_time = datetime.datetime.now()
                # compile the oracle
                oracle = compiler.compile(trial.problem)
                end_time = datetime.datetime.now()
                duration = (end_time - start_time).total_seconds()
                print(f"    Duration for problem {problem_id} with compiler {trial.compiler_name}: {duration}")

                trial.extra_data = trial.extra_data or {}
                trial.extra_data["compile_time"] = duration
                db.save_trial(trial)
    


        print(f"Finished processing {n} variables")

In [ ]:
collect_compile_times(oracle_db, n_iter=range(3,9), samples_per_n=100)